# 01. Разведочный анализ данных (EDA)

**Цель:** Первичный анализ данных о пожарах, оценка качества данных, выявление проблем.

**Вход:** Excel-файл с данными о пожарах

**Выход:**
- Отчёт о структуре данных
- Статистика пропусков
- Графики распределений
- Список проблемных записей

## 1. Инициализация

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Добавляем src в path
sys.path.insert(0, str(Path.cwd() / "src"))

from fire_es.schema import RU_COLS, EN_COLS, RU_TO_EN, FACT_SHEET_PREFIXES
from fire_es.cleaning import load_fact_sheet, sheet_period, clean_fire_data

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print("Инициализация завершена")

## 2. Загрузка данных

In [ ]:
# Поиск Excel-файла
xlsx_files = list(Path(".").glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("Excel-файл не найден в текущей директории")

DATA_PATH = xlsx_files[0]
print(f"Файл данных: {DATA_PATH.name}")

# Обзор листов
xl = pd.ExcelFile(DATA_PATH)
print(f"\\nЛисты в файле: {xl.sheet_names}")

# Информация о листах
sheet_info = []
for name in xl.sheet_names:
    try:
        df_tmp = xl.parse(name, nrows=0)
        rows = xl.parse(name).shape[0]
        cols = len(df_tmp.columns)
    except Exception as e:
        rows, cols = None, None
    sheet_info.append({"sheet": name, "rows": rows, "cols": cols})

pd.DataFrame(sheet_info)

## 3. Загрузка фактов

In [ ]:
# Определение факт-листов
fact_sheets = [s for s in xl.sheet_names if any(s.startswith(p) for p in FACT_SHEET_PREFIXES)]
print(f"Факт-листы: {fact_sheets}")

# Загрузка
raw_parts = [load_fact_sheet(s, xl) for s in fact_sheets]
raw_df = pd.concat(raw_parts, ignore_index=True)

# Добавление периода
raw_df["source_period"] = raw_df["source_sheet"].apply(sheet_period)

print(f"\\nВсего строк: {len(raw_df)}")
print(f"Колонок: {len(raw_df.columns)}")
raw_df.head(3)

## 4. Дедупликация

In [ ]:
# Поиск дубликатов
raw_df["dup_flag"] = raw_df.duplicated(subset=RU_COLS, keep="first")

print(f"Дубликатов: {raw_df['dup_flag'].sum()}")
print(f"После дедупликации: {(~raw_df['dup_flag']).sum()}")

raw_df_nodup = raw_df[~raw_df["dup_flag"]].copy()

# Распределение по листам
fig = px.bar(
    raw_df_nodup["source_sheet"].value_counts().reset_index(),
    x="source_sheet",
    y="count",
    title="Распределение по листам"
)
fig.show()

## 5. Статистика пропусков

In [ ]:
# Процент пропусков по колонкам
missing_pct = (raw_df_nodup.isna().sum() / len(raw_df_nodup) * 100).sort_values(ascending=False)

print("Top 20 пропусков:")
print(missing_pct.head(20))

# Визуализация
fig = px.bar(
    x=missing_pct.head(20).values,
    y=missing_pct.head(20).index,
    orientation="h",
    title="Top 20 колонок по пропускам (%)",
    labels={"x": "% пропусков", "y": "Колонка"}
)
fig.show()

## 6. Описательная статистика

In [ ]:
# Числовые колонки
num_cols = ["building_floors", "fire_floor", "distance_to_station", "direct_damage"]

# Временная статистика
print("Описательная статистика:")
print(raw_df_nodup[num_cols].describe())

# Распределение этажности
fig = px.histogram(
    raw_df_nodup,
    x="Этажность здания",
    nbins=50,
    title="Распределение этажности зданий",
    labels={"Этажность здания": "Количество этажей"}
)
fig.show()

## 7. Распределение по годам

In [ ]:
# Извлечение года из даты
raw_df_nodup["year"] = pd.to_datetime(raw_df_nodup["Дата возникновения пожара"], errors="coerce").dt.year

year_counts = raw_df_nodup["year"].value_counts().sort_index().reset_index()
year_counts.columns = ["year", "count"]

fig = px.line(
    year_counts,
    x="year",
    y="count",
    title="Количество пожаров по годам",
    markers=True
)
fig.show()

## 8. Сохранение артефактов

In [ ]:
# Сохранение сырых данных
output_dir = Path("data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

raw_df_nodup.to_parquet(output_dir / "raw_df.parquet", index=False)
print(f"Сохранено: {output_dir / 'raw_df.parquet'}")

# Сохранение статистики
import json
stats = {
    "total_rows": int(len(raw_df_nodup)),
    "duplicates_removed": int(raw_df["dup_flag"].sum()),
    "missing_pct_top20": missing_pct.head(20).to_dict(),
}

with open("reports/tables/eda_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)
print("Статистика сохранена в reports/tables/eda_stats.json")